# 🗄️ Big Data Batch Processing Pipeline (PySpark)

This notebook builds a batch data processing pipeline using **Apache Spark (PySpark)** to clean, transform, and analyze a large synthetic e-commerce transactions dataset (500,000+ records).

**Pipeline steps:** Generate data → Load into Spark → Clean → Aggregate → Window functions → Write to Parquet

In [3]:
!pip install pyspark -q

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataBatchPipeline") \
    .getOrCreate()

print("✅ Spark session started!")
spark

✅ Spark session started!


## Step 1: Generate a Large Synthetic Dataset

We simulate 500,000 e-commerce transactions, including some intentionally messy data (nulls and duplicates) to demonstrate real-world data cleaning.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n_rows = 500000

categories = ["Electronics", "Clothing", "Home & Kitchen", "Books", "Toys", "Sports", "Beauty"]
regions = ["North", "South", "East", "West"]
payment_methods = ["Credit Card", "Debit Card", "Cash on Delivery", "Wallet"]

start_date = datetime(2025, 1, 1)

data = {
    "transaction_id": range(1, n_rows + 1),
    "customer_id": np.random.randint(1000, 5000, n_rows),
    "category": np.random.choice(categories, n_rows),
    "region": np.random.choice(regions, n_rows),
    "payment_method": np.random.choice(payment_methods, n_rows),
    "quantity": np.random.randint(1, 10, n_rows),
    "unit_price": np.round(np.random.uniform(5, 500, n_rows), 2),
    "order_date": [start_date + timedelta(days=int(x)) for x in np.random.randint(0, 365, n_rows)]
}

pdf = pd.DataFrame(data)
pdf["total_amount"] = pdf["quantity"] * pdf["unit_price"]

# Introduce some messy data on purpose (nulls and duplicates) to simulate real-world data
pdf.loc[pdf.sample(frac=0.01, random_state=1).index, "unit_price"] = np.nan
pdf = pd.concat([pdf, pdf.sample(1000, random_state=2)], ignore_index=True)

print(f"Generated {len(pdf):,} rows")
pdf.head()

Generated 501,000 rows


,transaction_id,customer_id,category,region,payment_method,quantity,unit_price,order_date,total_amount
0,1,4174,Beauty,South,Debit Card,5,205.86,2025-01-19,1029.30
1,2,4507,Clothing,South,Credit Card,6,61.35,2025-03-30,368.10
2,3,1860,Electronics,North,Debit Card,2,344.46,2025-01-19,688.92
3,4,2294,Home & Kitchen,North,Credit Card,6,168.07,2025-11-28,1008.42
4,5,2130,Toys,North,Cash on Delivery,3,343.19,2025-09-02,1029.57


In [6]:
df = spark.createDataFrame(pdf)
df.printSchema()
print(f"Total records loaded into Spark: {df.count():,}")

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- total_amount: double (nullable = true)

Total records loaded into Spark: 501,000


## Step 2: Data Cleaning

Remove null values, duplicate transactions, and invalid records (e.g. zero or negative quantity/price).

In [7]:
from pyspark.sql.functions import col

print(f"Before cleaning: {df.count():,} rows")

df_clean = df.dropna(subset=["unit_price"])
df_clean = df_clean.dropDuplicates(["transaction_id"])
df_clean = df_clean.filter((col("quantity") > 0) & (col("unit_price") > 0))

print(f"After cleaning: {df_clean.count():,} rows")

Before cleaning: 501,000 rows
After cleaning: 495,000 rows


## Step 3: Aggregations — Revenue by Category

In [8]:
from pyspark.sql.functions import sum as _sum, count

revenue_by_category = df_clean.groupBy("category") \
    .agg(_sum("total_amount").alias("total_revenue"), count("transaction_id").alias("num_orders")) \
    .orderBy(col("total_revenue").desc())

revenue_by_category.show()

+--------------+-------------------+----------+
|      category|      total_revenue|num_orders|
+--------------+-------------------+----------+
|Home & Kitchen|8.978306840999943E7|     70788|
|         Books|8.935877301999977E7|     70983|
|          Toys|8.927487340000099E7|     70827|
|   Electronics|8.918395077000055E7|     70543|
|        Beauty|8.907864030999993E7|     70761|
|        Sports| 8.88826219400005E7|     70620|
|      Clothing|8.874833424999985E7|     70478|
+--------------+-------------------+----------+



## Step 4: Monthly Revenue Trend

In [9]:
from pyspark.sql.functions import month

df_clean = df_clean.withColumn("order_month", month("order_date"))

monthly_revenue = df_clean.groupBy("order_month") \
    .agg(_sum("total_amount").alias("monthly_revenue")) \
    .orderBy("order_month")

monthly_revenue.show(12)

+-----------+--------------------+
|order_month|     monthly_revenue|
+-----------+--------------------+
|          1|5.3172503420000225E7|
|          2| 4.811513598000024E7|
|          3| 5.298333256000009E7|
|          4| 5.110982560999991E7|
|          5|  5.25471013899999E7|
|          6| 5.061157950000012E7|
|          7| 5.343266572000013E7|
|          8| 5.306067392000008E7|
|          9|5.1347771169999935E7|
|         10| 5.315799565000025E7|
|         11| 5.151990581000001E7|
|         12| 5.325177136999987E7|
+-----------+--------------------+



## Step 5: Window Function — Top Customers by Spend

Demonstrates ranking with Spark's window functions, a core distributed-computing skill used heavily in real Data Engineering roles.

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

customer_spend = df_clean.groupBy("customer_id") \
    .agg(_sum("total_amount").alias("total_spent"))

window_spec = Window.orderBy(col("total_spent").desc())
top_customers = customer_spend.withColumn("spend_rank", rank().over(window_spec)) \
    .orderBy("spend_rank")

print("Top 10 customers by total spend:")
top_customers.show(10)

Top 10 customers by total spend:
+-----------+------------------+----------+
|customer_id|       total_spent|spend_rank|
+-----------+------------------+----------+
|       3471|         253822.41|         1|
|       1386|223174.97999999998|         2|
|       1654|220042.45999999996|         3|
|       2842|217686.21999999997|         4|
|       4584|         213053.06|         5|
|       2088|212988.82999999996|         6|
|       2724|212608.45999999996|         7|
|       2243|212009.40000000005|         8|
|       4845|210774.59000000003|         9|
|       1213|209593.13999999993|        10|
+-----------+------------------+----------+
only showing top 10 rows


## Step 6: Write Cleaned Data to Parquet

Parquet is the standard columnar storage format used in big data pipelines — much faster to read/write than CSV at scale.

In [11]:
output_path = "/content/output/cleaned_transactions"
df_clean.write.mode("overwrite").parquet(output_path)
print(f"✅ Cleaned data written to Parquet at: {output_path}")

verify_df = spark.read.parquet(output_path)
print(f"Verified record count in Parquet: {verify_df.count():,}")

✅ Cleaned data written to Parquet at: /content/output/cleaned_transactions
Verified record count in Parquet: 495,000
